# Notebook 4 – Demo tấn công và phòng thủ đối kháng

**Đề tài 2.21 – Nghiên cứu và phòng thủ trước tấn công đối kháng lên mô hình phân loại ảnh**

Notebook trình diễn trực tiếp, **không huấn luyện**: chỉ nạp 3 checkpoint đã có và chạy tấn công trên vài ảnh. Mỗi cell chạy trong vài giây trên CPU.

| Mục | Nội dung |
|---|---|
| 1 | Baseline đoán đúng ảnh sạch |
| 2 | PGD ε = 8/255 làm baseline đoán sai; nhiễu gần như không nhìn thấy |
| 3 | Targeted: buộc mô hình đoán thành lớp do mình chọn |
| 4 | Adversarial Training: tấn công trực tiếp mô hình PGD-AT vẫn không lừa được |
| 5 | Tiền xử lý JPEG: tưởng chống được, nhưng kẻ tấn công thích nghi (BPDA) vẫn vượt qua |
| 6 | Bảng và biểu đồ tổng hợp trên 1.000 ảnh (từ `results/`) |

Tập 1.000 ảnh đánh giá được đọc từ `assets/eval_subset_seed42.npz` (~3 MB) nên **không cần tải CIFAR-10**.

Cần có trong `checkpoints/` (hoặc `../AdvML_Project/checkpoints/`): `baseline_resnet18.pth`, `adv_pgd_resnet18.pth`, tùy chọn `adv_fast_resnet18.pth`.

## 0. Chuẩn bị

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    Path("/content/drive/MyDrive/AdvML_Project").mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = next(
        (root for root in (cwd, *cwd.parents)
         if (root / "src/config.py").is_file() and (root / "notebooks").is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise FileNotFoundError("Hãy mở notebook từ thư mục repo hoặc notebooks/.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display

from src import config
from src.attacks import fgsm, pgd
from src.data import get_eval_subset
from src.defenses import PreprocessedModel, jpeg_compress
from src.models import build_model
from src.utils import load_checkpoint, set_seed

device = config.get_device()
if device.type == "cpu":
    torch.set_num_threads(min(torch.get_num_threads(), 4))
EPS = 8 / 255
print("Device:", device)

In [ ]:
PROJECT_DIRS = [config.DRIVE_PROJECT_DIR] if IN_COLAB else [REPO_DIR.parent / "AdvML_Project"]


def find_checkpoint(name, required=True):
    """Tìm checkpoint trong repo/checkpoints trước, sau đó trong AdvML_Project/checkpoints."""
    candidates = [config.CHECKPOINT_DIR / name, *(root / "checkpoints" / name for root in PROJECT_DIRS)]
    path = next((p for p in candidates if p.is_file()), None)
    if path is None and required:
        raise FileNotFoundError(f"Không tìm thấy {name}. Đã kiểm tra:\n" + "\n".join(map(str, candidates)))
    return path


def find_data_dir():
    candidates = [config.DATA_DIR, *(root / "cifar-10-python" for root in PROJECT_DIRS),
                  *(root / "data" for root in PROJECT_DIRS), *PROJECT_DIRS]
    return next((p for p in candidates if (p / "cifar-10-batches-py/test_batch").is_file()), config.DATA_DIR)


# Ưu tiên bản lưu sẵn 1.000 ảnh đánh giá (~3 MB, đúng từng pixel với get_eval_subset, seed 42)
# để không phải tải toàn bộ CIFAR-10 (170 MB) trong buổi demo.
EVAL_NPZ = REPO_DIR / "assets" / "eval_subset_seed42.npz"
if EVAL_NPZ.is_file():
    _d = np.load(EVAL_NPZ)
    x_eval = torch.from_numpy(_d["x"]).float() / 255.0
    y_eval = torch.from_numpy(_d["y"]).long()
    print(f"Tập đánh giá: {EVAL_NPZ.name} ({len(y_eval)} ảnh)")
else:
    DATA_DIR = find_data_dir()
    x_eval, y_eval = get_eval_subset(
        as_tensors=True, data_dir=DATA_DIR,
        download=not (DATA_DIR / "cifar-10-batches-py/test_batch").is_file(),
        num_workers=config.NUM_WORKERS if device.type == "cuda" else 0,
    )


def load_model(name, required=True):
    path = find_checkpoint(name, required)
    if path is None:
        return None
    m = build_model().to(device)
    load_checkpoint(path, m, map_location=device)
    m.eval().requires_grad_(False)
    print("Đã nạp", path)
    return m


baseline = load_model(config.BASELINE_CKPT)
pgd_at = load_model(config.ADV_CKPT_PGD)
fast_at = load_model(config.ADV_CKPT_FAST, required=False)
MODELS = {"Baseline": baseline, "PGD-AT": pgd_at, **({"Fast-AT": fast_at} if fast_at is not None else {})}

In [ ]:
CLASSES = config.CLASSES


@torch.no_grad()
def probs_of(m, x):
    return m(x.to(device)).softmax(1).cpu()


def top_label(p):
    c = int(p.argmax())
    return f"{CLASSES[c]} {100 * p[c]:.1f}%"


def show_attack(x0, x_adv, model, title, true_label):
    """Ảnh gốc | nhiễu phóng đại | ảnh đối kháng, kèm dự đoán và độ tin cậy."""
    p0, pa = probs_of(model, x0[None])[0], probs_of(model, x_adv[None])[0]
    delta = x_adv - x0
    noise = (delta / (2 * delta.abs().max().clamp_min(1e-12)) + 0.5).clamp(0, 1)
    ok = lambda p: "✓" if int(p.argmax()) == true_label else "✗"
    fig, axes = plt.subplots(1, 3, figsize=(8.5, 3.2))
    panels = [(x0, f"Ảnh gốc\n{top_label(p0)} {ok(p0)}"),
              (noise, f"Nhiễu × phóng đại\nL∞ = {delta.abs().max() * 255:.1f}/255"),
              (x_adv, f"Ảnh đối kháng\n{top_label(pa)} {ok(pa)}")]
    for ax, (img, t) in zip(axes, panels):
        ax.imshow(img.permute(1, 2, 0).numpy(), interpolation="nearest")
        ax.set_title(t, fontsize=10); ax.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout(); plt.show()


def attack_one(m, x, y, targeted=False, steps=config.PGD_STEPS):
    set_seed(config.SEED)
    return pgd(m, x[None].to(device), torch.tensor([y], device=device), eps=EPS,
               alpha=config.PGD_ALPHA, steps=steps, random_start=True, targeted=targeted)[0].cpu()

## 1. Chọn ảnh demo

Notebook tìm ảnh đầu tiên thuộc `DEMO_CLASS` thỏa **mọi** tình huống của kịch bản: baseline đoán đúng với độ tin cậy ≥ 90%, bị PGD đánh lừa và bị targeted ép thành `TARGET`; PGD-AT và Fast-AT vẫn đúng dưới PGD-20; JPEG q=50 đoán đúng ảnh sạch và ảnh đối kháng chuyển từ baseline, nhưng bị BPDA vượt qua.
Đặt `DEMO_INDEX` (0–999) để dùng một ảnh cụ thể.

In [ ]:
DEMO_CLASS = "cat"
TARGET = "dog"
DEMO_INDEX = 65     # ảnh cat thỏa mọi tình huống (đã tìm sẵn); đặt None để tự tìm lại khi đổi DEMO_CLASS/TARGET

jpeg = partial(jpeg_compress, quality=50)
defended = PreprocessedModel(baseline, jpeg).eval()          # kẻ tấn công không biết bước JPEG
adaptive = PreprocessedModel(baseline, jpeg, bpda=True).eval()  # kẻ tấn công thích nghi (BPDA)


def pred(m, x):
    return int(probs_of(m, x[None])[0].argmax())


def fits_scenario(i):
    x, y = x_eval[i], int(y_eval[i])
    p = probs_of(baseline, x[None])[0]
    if int(p.argmax()) != y or p[y] < 0.9:
        return False
    if any(pred(m, x) != y for m in MODELS.values()) or pred(defended, x) != y:
        return False
    x_adv = attack_one(baseline, x, y)
    if pred(baseline, x_adv) == y or pred(defended, x_adv) != y:
        return False
    if pred(baseline, attack_one(baseline, x, CLASSES.index(TARGET), targeted=True)) != CLASSES.index(TARGET):
        return False
    if any(pred(m, attack_one(m, x, y, steps=20)) != y for m in (pgd_at, fast_at) if m is not None):
        return False
    return pred(defended, attack_one(adaptive, x, y)) != y


if DEMO_INDEX is None:
    clean_ok = torch.stack([probs_of(m, x_eval).argmax(1).eq(y_eval) for m in MODELS.values()]).all(0)
    candidates = torch.where(clean_ok & (y_eval == CLASSES.index(DEMO_CLASS)))[0].tolist()
    DEMO_INDEX = next((i for i in candidates if fits_scenario(i)), candidates[0])
x0, y0 = x_eval[DEMO_INDEX], int(y_eval[DEMO_INDEX])
print(f"Ảnh #{DEMO_INDEX} – nhãn thật: {CLASSES[y0]}")
for name, m in MODELS.items():
    print(f"  {name:<8s} dự đoán: {top_label(probs_of(m, x0[None])[0])}")
print(f"  JPEG q=50 + baseline dự đoán: {top_label(probs_of(defended, x0[None])[0])}")
plt.figure(figsize=(2.5, 2.5)); plt.imshow(x0.permute(1, 2, 0).numpy(), interpolation="nearest")
plt.title(CLASSES[y0]); plt.axis("off"); plt.show()

## 2. Tấn công PGD lên baseline (untargeted, ε = 8/255)

Nhiễu tối đa 8/255 ≈ 3% mỗi pixel. Cột giữa phóng đại nhiễu để nhìn thấy; hai ảnh hai bên dùng pixel thật.

In [ ]:
x_adv_base = attack_one(baseline, x0, y0)
show_attack(x0, x_adv_base, baseline, "Baseline bị PGD-10 đánh lừa", y0)

## 3. Tấn công có mục tiêu: buộc baseline đoán thành lớp tùy chọn

In [ ]:
if TARGET == CLASSES[y0]:
    TARGET = "cat" if CLASSES[y0] != "cat" else "dog"
x_adv_tgt = attack_one(baseline, x0, CLASSES.index(TARGET), targeted=True)
show_attack(x0, x_adv_tgt, baseline, f"Targeted PGD-10: {CLASSES[y0]} → {TARGET}", y0)

## 4. Phòng thủ bằng Adversarial Training

PGD-AT và Fast-AT có **cùng kiến trúc ResNet-18** với baseline; khác ở chỗ khi huấn luyện, mỗi batch được tấn công trước (PGD-7 hoặc FGSM) rồi mới học trên ảnh đối kháng, tức bài toán min–max của Madry. Ở đây em tấn công **trực tiếp** từng mô hình với cùng ngân sách (white-box, kẻ tấn công biết toàn bộ trọng số), dùng PGD-20 mạnh hơn lúc huấn luyện.

In [ ]:
x_adv_at = attack_one(pgd_at, x0, y0, steps=20)
show_attack(x0, x_adv_at, pgd_at, "PGD-AT bị tấn công trực tiếp bằng PGD-20", y0)

if fast_at is not None:
    x_adv_fast = attack_one(fast_at, x0, y0, steps=20)
    show_attack(x0, x_adv_fast, fast_at, "Fast-AT bị tấn công trực tiếp bằng PGD-20", y0)

print("Ảnh đối kháng sinh trên baseline (mục 2) đưa vào từng mô hình:")
for name, m in MODELS.items():
    print(f"  {name:<8s} {top_label(probs_of(m, x_adv_base[None])[0])}")

## 5. Tiền xử lý JPEG và cái bẫy gradient masking

- **Kẻ tấn công không biết phòng thủ:** ảnh đối kháng của mục 2 đi qua JPEG q=50 rồi mới vào baseline → nhiễu bị "xoá", mô hình đoán đúng trở lại.
- **Kẻ tấn công thích nghi (BPDA):** biết có bước JPEG, tấn công thẳng vào `baseline(JPEG(x))` (lượt backward coi JPEG ≈ hàm đồng nhất) → bị lừa lần nữa.

In [ ]:
show_attack(x0, x_adv_base, defended, "JPEG q=50 trước baseline – kẻ tấn công KHÔNG biết phòng thủ", y0)
x_adv_bpda = attack_one(adaptive, x0, y0)
show_attack(x0, x_adv_bpda, defended, "JPEG q=50 trước baseline – kẻ tấn công thích nghi (BPDA)", y0)

## 6. Số liệu tổng hợp trên 1.000 ảnh

Lấy từ các file trong `results/` (đã đo ở Notebook 2, 3a, 3b; bảng do `scripts/make_summary_figures.py` tạo).

In [ ]:
for name in ["summary_robustness_eps.png", "summary_tradeoff.png", "preprocessing_accuracy.png"]:
    path = config.RESULTS_DIR / name
    if path.exists():
        display(Image(filename=str(path), width=900))
table = config.RESULTS_DIR / "summary_comparison.md"
if table.exists():
    display(Markdown(table.read_text(encoding="utf-8")))

## (Tùy chọn) Đo trực tiếp trên N ảnh

Đặt `RUN_LIVE = True` để đo lại accuracy dưới PGD-10 ngay trong buổi demo (N = 200 ảnh mất khoảng 1–2 phút trên CPU).

In [ ]:
RUN_LIVE = False
N_LIVE = 200

if RUN_LIVE:
    xs, ys = x_eval[:N_LIVE], y_eval[:N_LIVE]
    for name, m in MODELS.items():
        set_seed(config.SEED)
        clean = probs_of(m, xs).argmax(1).eq(ys).float().mean() * 100
        x_adv = pgd(m, xs.to(device), ys.to(device), eps=EPS, alpha=config.PGD_ALPHA, steps=config.PGD_STEPS)
        rob = probs_of(m, x_adv).argmax(1).eq(ys).float().mean() * 100
        print(f"{name:<8s} sạch {clean:5.1f}% | PGD-10 8/255 {rob:5.1f}%")